# Week 2 — SQL Analysis & Deeper Insights

Walmart Sales Project · Team Notebook

**Workflow reminder:** branch → commit → push → PR → merge (same as Week 1's EDA notebook).

This notebook combines everyone's work into one place:
- **Person A** — core SQL queries (`src/queries.py`)
- **Person B** — time-series / trend analysis
- **Person C** — correlation + external factors analysis

Fill in your section, run it end-to-end, then push your branch for review.


In [ ]:
import sys
sys.path.append("..")  # so we can import src/queries.py

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

from src.queries import (
    total_sales_by_store, avg_sales_holiday_vs_normal, top_stores, worst_stores,
    monthly_sales_trend, yearly_sales_trend, store_monthly_trend,
    holiday_sensitivity_by_store, store_performance_tiers, load_full_table
)

conn = sqlite3.connect("../db/walmart.db")
pd.read_sql("SELECT COUNT(*) AS row_count FROM Sales", conn)


---
## 1. Foundational SQL Queries  *(Person A)*

Core queries live in `src/queries.py` so anyone on the team can reuse them without
retyping SQL. Below: total sales per store, holiday vs non-holiday averages, and
top/bottom performers.


In [ ]:
# Total sales per store
store_totals = total_sales_by_store(conn)
store_totals.head(10)


In [ ]:
# Average weekly sales, holiday vs non-holiday
avg_sales_holiday_vs_normal(conn)


In [ ]:
# Top 5 performing stores
top_stores(conn, n=5)


In [ ]:
# Bottom 5 performing (worst) stores
worst_stores(conn, n=5)


In [ ]:
# Quick bar chart: total sales by store
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(store_totals["store_id"].astype(str), store_totals["total_sales"])
ax.set_xlabel("Store ID")
ax.set_ylabel("Total Sales")
ax.set_title("Total Sales by Store")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


---
## 2. Trend Analysis Over Time  *(Person B)*

- Monthly / quarterly sales trend
- Year-over-year comparison (2010 vs 2011 vs 2012)
- Which stores are growing vs declining


In [ ]:
# Monthly sales trend (all stores combined)
monthly_trend = monthly_sales_trend(conn)
monthly_trend


In [ ]:
# Line chart: monthly trend
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_trend["month"], monthly_trend["total_sales"], marker="o")
ax.set_xlabel("Month")
ax.set_ylabel("Total Sales")
ax.set_title("Monthly Sales Trend (All Stores)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# Year-over-year comparison
yearly_trend = yearly_sales_trend(conn)
yearly_trend


In [ ]:
# TODO (Person B): growth vs decline by store
# Idea: pull store_monthly_trend(conn) for all stores, then compare each store's
# first-half vs second-half average (or fit a simple linear trend per store)
# to rank stores as "growing" vs "declining".

all_store_trend = store_monthly_trend(conn)
all_store_trend.head()


---
## 3. Correlation & External Factors  *(Person C)*

Does `temperature`, `unemployment`, `fuel_price`, or `cpi` correlate with `weekly_sales`?


In [ ]:
# Load full table for correlation analysis
df = load_full_table(conn)
df.head()


In [ ]:
# Correlation matrix
numeric_cols = ["weekly_sales", "temperature", "fuel_price", "cpi", "unemployment", "holiday_flag"]
corr_matrix = df[numeric_cols].corr()
corr_matrix


In [ ]:
# Heatmap of correlations
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im)
ax.set_title("Correlation Matrix: Sales vs External Factors")
plt.tight_layout()
plt.show()


In [ ]:
# TODO (Person C): interpret findings
# Which factor(s) show the strongest (positive/negative) correlation with weekly_sales?
# Are any correlations misleading (e.g. confounded by holiday_flag or store size)?


---
## 4. Store Segmentation

- Performance tiers (High / Medium / Low sales)
- Holiday-sensitive stores (biggest jump when `holiday_flag == 1`)


In [ ]:
# Store performance tiers
tiers = store_performance_tiers(conn)
tiers


In [ ]:
# Holiday sensitivity by store (sorted by % lift, most sensitive first)
sensitivity = holiday_sensitivity_by_store(conn)
sensitivity.head(10)


---
## 5. Summary of Findings

_Fill in as a team after all sections are complete:_

- **Top performing stores:** ...
- **Weakest performing stores:** ...
- **Seasonal / yearly trend:** ...
- **Strongest correlation with sales:** ...
- **Most holiday-sensitive stores:** ...
- **Performance tier breakdown:** ...

---
**Next steps (Week 3+):** turn these findings into a dashboard.


In [ ]:
conn.close()